Hands-on Task 3: Three-Way Conditional Router 

Objective 

Implement a routing system with multiple decision branches. 

Task 

Create a router that classifies user questions into: 
science 
history 
general 
Based on this classification, route the input to one of three nodes: 
science_node 
history_node 
general_node 
`` 
Node Behavior 
science_node → Respond as a science teacher 
history_node → Respond as a historian 
general_node → Respond as a general assistant 
Routing Requirement 
The routing function must return: 
Literal["science_node", "history_node", "general_node"] 
Testing Requirement 
Test with at least 4 questions, ensuring all categories are covered. 
Skills Tested 
Using add_conditional_edges 
Implementing multi-branch routing 
Applying Literal type hints 
Designing persona-based responses 
 
Expected Outcome 
By completing these tasks, you will be able to: 
Extend pipeline state using TypedDict 
Design multi-node workflows 
Build LLM-powered pipelines 
Route inputs dynamically across multiple branches 
Apply persona-based response strategies 
 
 

In [1]:
from dotenv import load_dotenv
import os
from typing import TypedDict, Literal
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

# ==========================================
# 1. Load Custom Environment Variables
# ==========================================
load_dotenv()

api_key = os.environ.get("KEY")
base_url = os.environ.get("BASE_URL")
model_name = os.environ.get("MODEL", "global.anthropic.claude-haiku-4-5-20251001-v1:0")

if not api_key:
    raise ValueError("❌ 'KEY' not found in your environment or .env file.")

# Initialize the custom LLM
llm = ChatAnthropic(
    model=model_name,
    anthropic_api_key=api_key,
    anthropic_api_url=base_url,
    temperature=0.7
)

# ==========================================
# 2. Define the Router State
# ==========================================
class RouterState(TypedDict):
    question: str
    category: str
    response: str

# ==========================================
# 3. Define the Router Logic
# ==========================================
def route_question(state: RouterState) -> Literal["science_node", "history_node", "general_node"]:
    """Classifies the question and determines the next node to execute."""
    prompt = (
        f"Classify this question into exactly one of three categories: science, history, or general.\n\n"
        f"Question: {state['question']}\n\n"
        f"Respond with only the category name word (science, history, or general). Do not include formatting or punctuation."
    )
    # Force low temperature for deterministic classification
    classifier = llm.with_config(configurable={"temperature": 0.0})
    category = classifier.invoke(prompt).content.strip().lower()
    
    # Map the classification to the correct target node execution name
    if "science" in category:
        return "science_node"
    elif "history" in category:
        return "history_node"
    else:
        return "general_node"

# ==========================================
# 4. Define Persona-Based Nodes
# ==========================================

def science_node(state: RouterState) -> dict:
    """Responds as an enthusiastic science teacher."""
    prompt = f"You are an enthusiastic science teacher. Answer this question clearly with educational flair: {state['question']}"
    res = llm.invoke(prompt)
    return {"response": res.content, "category": "science"}

def history_node(state: RouterState) -> dict:
    """Responds as a deeply knowledgeable historian."""
    prompt = f"You are a deeply knowledgeable historian. Answer this question providing rich context and historical perspective: {state['question']}"
    res = llm.invoke(prompt)
    return {"response": res.content, "category": "history"}

def general_node(state: RouterState) -> dict:
    """Responds as a polite, direct general assistant."""
    prompt = f"You are a helpful and polite general assistant. Answer this question directly: {state['question']}"
    res = llm.invoke(prompt)
    return {"response": res.content, "category": "general"}

# ==========================================
# 5. Build Graph with Conditional Edges
# ==========================================
workflow = StateGraph(RouterState)

# Add our processing branches
workflow.add_node("science_node", science_node)
workflow.add_node("history_node", history_node)
workflow.add_node("general_node", general_node)

# START directs immediately to the conditional router function
workflow.add_conditional_edges(
    START,
    route_question,
    {
        "science_node": "science_node",
        "history_node": "history_node",
        "general_node": "general_node"
    }
)

# All execution branches flow into the END node
workflow.add_edge("science_node", END)
workflow.add_edge("history_node", END)
workflow.add_edge("general_node", END)

# Compile graph
app = workflow.compile()

# ==========================================
# 6. Run Testing Suite (4 Questions)
# ==========================================
test_questions = [
    "Why is the sky blue and how does scattering work?",
    "Who was the first emperor of Rome and when did he rule?",
    "What is the best way to organize a daily grocery shopping list?",
    "What happens to water molecules when they freeze into ice?"
]

print(" Running Router Pipeline Testing...\n")

for i, q in enumerate(test_questions, start=1):
    result = app.invoke({"question": q})
    
    print(f"--- Test Case {i} ---")
    print(f"Question: {result['question']}")
    print(f"Routed To: {result['category'].upper()}_NODE")
    print(f"Response:\n{result['response']}\n")

 Running Router Pipeline Testing...

--- Test Case 1 ---
Question: Why is the sky blue and how does scattering work?
Routed To: SCIENCE_NODE
Response:
# Why the Sky is Blue! 🌤️

Oh, what a *fantastic* question! This is one of my favorite demonstrations of physics in action!

## The Short Answer
The sky is blue because of **Rayleigh scattering**—a process where light bounces off tiny molecules in our atmosphere, and blue light scatters more than other colors!

## How Scattering Works (The Cool Part!)

Imagine sunlight as a bundle of different colored waves traveling through space. When this light hits molecules in our air (nitrogen, oxygen, etc.), something amazing happens:

**The molecules act like tiny mirrors and scatter the light in all directions!**

### Why Blue Specifically?

Here's where it gets really interesting:

- 🔴 **Red light** has long waves → passes mostly straight through
- 🟡 **Yellow light** scatters a bit
- 🔵 **Blue light** has short waves → scatters MUCH MORE!

**Th